# Chapter 3 — Is an overdensity made of stars?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/03_star_galaxy_populations.ipynb)

**Goal.** Chapter 1 showed that faint "stars" are partly galaxies. Galaxies cluster, so a group or cluster of
galaxies produces a compact overdensity of "stars" that looks exactly like a faint satellite on a sky map. Here
we compare two overdensities: the galaxy cluster **Abell 370** (redshift z = 0.375) and the dwarf galaxy
**Boötes I**. Both are highly significant excesses of SDSS point sources. Our question is about the *excess
population* — not about any single object.

**You will learn**
1. to test an overdensity in surveys of different depth and image quality;
2. to describe the excess in colour–colour space as a mixture of a stellar template and a galaxy template, and
   measure its **star fraction** $f_\star$ with a likelihood;
3. how photometric errors, and their correlations between colours, enter such templates;
4. to compare the shape (PSF − model magnitude) of the excess objects with that of real stars of the same magnitude.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


def lsmag(flux):
    """Legacy Surveys fluxes are in nanomaggies: m = 22.5 - 2.5 log10(flux)."""
    return 22.5 - 2.5 * np.log10(np.where(flux > 0, flux, np.nan))


A370 = (39.970, -1.577)

## 3.1 The same field in four surveys

For each survey we keep what *that survey* calls a star (the classifiers of Chapter 1) with 14 < r < 23, and
measure the surface density in rings around the cluster centre, relative to the density at 8′–20′.

In [ ]:
surveys = {}
d = load('a370_sdss'); surveys['SDSS'] = (d, d.psfmag_r.to_numpy(), (d.type == 6).to_numpy())
d = load('a370_ps1'); surveys['PS1'] = (d, d.rpsfmag.to_numpy(), (d.rpsfmag - d.rkronmag < 0.05).to_numpy())
d = load('a370_ls'); surveys['LS'] = (d, lsmag(d.flux_r.to_numpy()), (d.type == 'PSF').to_numpy())
d = load('a370_hsc'); surveys['HSC'] = (d, d.r_psfflux_mag.to_numpy(), (d.i_extendedness_value < 0.5).to_numpy())

edges = np.array([0, 1, 2, 4, 8, 19.8])                          # arcmin
area = np.pi * np.diff(edges**2)
counts, inner = {}, {}
fig, ax = plt.subplots(figsize=(7, 4.5))
for k, (d, m, cl) in surveys.items():
    R = np.hypot(*tangent(d.ra.to_numpy(), d.dec.to_numpy(), *A370)) * 60
    s = cl & (m > 14) & (m < 23)
    n = np.histogram(R[s], edges)[0]
    bg = n[-1] / area[-1]                                        # stars per arcmin^2 at 8'-20'
    counts[k] = n
    inner[k] = (n[0] + n[1], bg * np.pi * 2**2)                  # observed and expected inside 2'
    ax.errorbar(0.5 * (edges[1:] + edges[:-1]), n / area / bg, np.sqrt(np.maximum(n, 1)) / area / bg, marker='o',
                label=f'{k}: {bg * 3600:,.0f} "stars" per deg²')
ax.axhline(1, c='k', lw=0.6)
ax.set(xscale='log', xlabel="distance from Abell 370 [arcmin]", ylabel='density / background', title='point sources, 14 < r < 23')
ax.legend(fontsize=8)
plt.tight_layout()

for k, (n, b) in inner.items():
    p = stats.poisson.sf(n - 1, b)                               # P(N >= n | background b)
    up = stats.chi2.ppf(0.95, 2 * (n + 1)) / 2                   # 95% upper limit on the Poisson mean, given n
    print(f'{k:5s}: {n:3d} inside 2′, background {b:5.1f}; P(N >= n) = {p:.1e}; 95% upper limit on ratio {up / b:.2f}')
assert stats.poisson.sf(inner['SDSS'][0] - 1, inner['SDSS'][1]) < 1e-6
n, b = inner['HSC']
assert stats.chi2.ppf(0.95, 2 * (n + 1)) / 2 / b < 2

**Figure 3.1.** SDSS and PS1 see a strong central excess of point sources — by the Poisson probability alone, a
"detection" far beyond 5σ. LS, built from deeper DECam images with a better model fit, shows a weak excess; HSC
(0.6″ seeing, 8-m telescope) shows none. Note also the background densities: at the same magnitude limit the
surveys disagree by a factor of several on how many "stars" there are, because their contamination and
completeness differ (Chapter 1).

The lesson is not "use the deeper survey" — often there is none. It is that we need tests that can be applied
to the excess *in the survey we have*.

## 3.2 The colours of the excess: a two-template likelihood

Bin the objects of a region in colour–colour space $(g-r, r-i)_0$. In the on-region (the overdensity), the expected
counts in bin $j$ are

$$ \lambda_j = B_j + n_\star T^\star_j + n_G T^G_j, $$

where $B_j = \alpha N^{\rm off}_j$ is the background predicted from the off-region (scaled by the area ratio α),
$T^\star$ and $T^G$ are normalised templates for stars and galaxies, and $n_\star, n_G \ge 0$ are the numbers of
excess stars and galaxies. The Poisson log-likelihood of the observed counts $N_j$ is
$\ln\mathcal L = \sum_j (N_j \ln\lambda_j - \lambda_j)$. The star fraction is $f_\star = n_\star/(n_\star + n_G)$; we
find its 68% interval from the profile likelihood ($2\Delta\ln\mathcal L < 1$, maximising over $n_\star + n_G$).

**Templates, from COSMOS** (Chapter 1), in SDSS:
- galaxy template: SDSS "stars" with 20 < r < 22.5 that HST shows to be galaxies — exactly the contaminants;
- stellar template: bright HST-confirmed stars (16 < r < 20, small errors), each given random noise drawn from
  the errors of the *target* stars. Noise is added to g, r and i separately, so the colours inherit the right
  correlation: an error in r moves g − r and r − i in opposite directions, ${\rm Cov}(g-r, r-i) = -\sigma_r^2$.

In [ ]:
rng = np.random.default_rng(3)
hst = load('cosmos_hst')
hst = hst[hst.mu_class.isin([1, 2])]
cs = load('cosmos_sdss')
c = np.cos(np.radians(2.2))
dist, idx = cKDTree(np.c_[hst.ra * c, hst.dec]).query(np.c_[cs.ra * c, cs.dec], distance_upper_bound=1 / 3600)
matched = np.isfinite(dist)
hst_star = np.zeros(len(cs), bool)
hst_star[matched] = hst.mu_class.to_numpy()[idx[matched]] == 2


def dered(d):
    """Dereddened g, r, i and their errors."""
    return [(d[f'psfmag_{b}'] - d[f'extinction_{b}']).to_numpy() for b in 'gri'], [d[f'psfmagerr_{b}'].to_numpy() for b in 'gri']


gb, ib = np.arange(-0.5, 2.51, 0.2), np.arange(-0.5, 2.01, 0.2)
hist = lambda gr, ri: np.histogram2d(gr, ri, [gb, ib])[0]
(Gc, Rc, Ic), _ = dered(cs)
point = (cs.type == 6).to_numpy() & matched
faint = (Rc > 20) & (Rc < 22.5)
Tg = gaussian_filter(hist((Gc - Rc)[point & ~hst_star & faint], (Rc - Ic)[point & ~hst_star & faint]), 0.7) + 1e-3
Tg /= Tg.sum()
bright = point & hst_star & (Rc > 16) & (Rc < 20)


def star_template(errs, n_copies=20):
    """Bright HST stars with noise drawn from the target's errors (errs = [sigma_g, sigma_r, sigma_i] arrays)."""
    k = rng.integers(0, len(errs[0]), size=(n_copies, bright.sum()))
    g, r, i = [m[bright] + rng.normal(size=k.shape) * e[k] for m, e in zip((Gc, Rc, Ic), errs)]
    T = hist((g - r).ravel(), (r - i).ravel()) + 1e-3
    return T / T.sum()


def fit_fstar(on, B, Ts):
    """Best (n_star, n_gal) and the profile likelihood of f_star = n_star / (n_star + n_gal)."""
    lnl = lambda ns, ng: np.sum(on * np.log(B + ns * Ts + ng * Tg + 1e-12) - (B + ns * Ts + ng * Tg))
    best = minimize(lambda n: -lnl(*n), [10, 10], method='L-BFGS-B', bounds=[(0, None)] * 2).x
    fs = np.linspace(0, 1, 101)
    prof = np.array([-minimize(lambda t: -lnl(t[0] * f, t[0] * (1 - f)), [best.sum()], method='L-BFGS-B',
                               bounds=[(0, None)]).fun for f in fs])
    prof = 2 * (prof.max() - prof)
    inside = fs[prof < 1]
    return best, fs, prof, (inside.min(), inside.max())


def region(name, centre, r_on, r_off, point_only):
    d = load(name)
    if point_only:
        d = d[d.type == 6]
    (g, r, i), e = dered(d)
    R = np.hypot(*tangent(d.ra.to_numpy(), d.dec.to_numpy(), *centre)) * 60
    s = (r > 20) & (r < 22.5)
    on, off = s & (R < r_on), s & (R > r_off[0]) & (R < r_off[1])
    alpha = r_on**2 / (r_off[1]**2 - r_off[0]**2)
    N_on, B = hist((g - r)[on], (r - i)[on]), alpha * hist((g - r)[off], (r - i)[off])
    Ts = star_template([x[on] for x in e])
    return N_on, B, Ts, (g - r)[on], (r - i)[on], (g - r)[off], (r - i)[off]


lvdb = load('lvdb').set_index('key')
boo = lvdb.loc['bootes_1']
cases = {'Abell 370 (R < 2′)': region('a370_sdss', A370, 2, (8, 19.8), True),
         f'Boötes I (R < r_h)': region('bootes1_sdss', (boo.ra, boo.dec), boo.rhalf, (60, 120), False)}
results = {}
for k, (N_on, B, Ts, *_) in cases.items():
    best, fs, prof, ci = fit_fstar(N_on, B, Ts)
    results[k] = (best, fs, prof, ci)
    print(f'{k:20s}: on {N_on.sum():4.0f}, background {B.sum():6.1f}, excess {N_on.sum() - B.sum():6.1f}; '
          f'fit n_star = {best[0]:6.1f}, n_gal = {best[1]:5.1f}; f_star 68% interval [{ci[0]:.2f}, {ci[1]:.2f}]')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ext = [gb[0], gb[-1], ib[0], ib[-1]]
ax[0].contour(0.5 * (gb[1:] + gb[:-1]), 0.5 * (ib[1:] + ib[:-1]), Ts.T, levels=np.array([0.005, 0.02, 0.05]), colors='C0')
ax[0].contour(0.5 * (gb[1:] + gb[:-1]), 0.5 * (ib[1:] + ib[:-1]), Tg.T, levels=np.array([0.005, 0.02, 0.05]), colors='C1')
for (k, v), mk, cc in zip(cases.items(), ['s', 'o'], ['k', 'C2']):
    ax[0].scatter(v[3], v[4], s=10, marker=mk, c=cc, label=f'{k}: on-region')
ax[0].set(xlim=(-0.5, 2.3), ylim=(-0.5, 1.8), xlabel='$(g-r)_0$', ylabel='$(r-i)_0$',
          title='templates: stars (blue), galaxies (orange)')
ax[0].legend(fontsize=7)
for a, (k, (N_on, B, *_)) in zip(ax[1:], cases.items()):
    im = a.imshow((N_on - B).T, origin='lower', extent=ext, cmap='RdBu_r', vmin=-15, vmax=15, aspect='auto')
    a.set(xlabel='$(g-r)_0$', ylabel='$(r-i)_0$', title=f'on − off: {k}', xlim=(-0.5, 2.3), ylim=(-0.5, 1.8))
    plt.colorbar(im, ax=a)
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 4))
for k, (best, fs, prof, ci) in results.items():
    ax.plot(fs, prof, label=k)
ax.axhline(1, c='k', ls=':', lw=0.8)
ax.set(xlabel='star fraction of the excess $f_\\star$', ylabel='$2\\Delta\\ln\\mathcal{L}$', ylim=(0, 20))
ax.legend()
plt.tight_layout()

(ca, cb_) = results['Abell 370 (R < 2′)'][3], results['Boötes I (R < r_h)'][3]
assert cb_[0] > 0.7 and ca[1] < 0.3 and ca[1] < cb_[0]

**Figure 3.2.** Left: the two templates (blue: stars blurred by the errors of faint targets; orange: galaxies
that SDSS calls stars) and the objects in the two on-regions. Middle and right: on − off counts. The Abell 370
excess lies at g − r ≈ 1.4–2.0 and r − i ≈ 0.3–0.7, *below* the M-dwarf branch of the stellar locus: the
red sequence of the cluster's early-type galaxies. The Boötes I excess lies on the blue part of the stellar locus,
where old, metal-poor stars are.

**Figure 3.3.** Profile likelihood of the star fraction. The two intervals do not overlap: the Abell 370 excess
is consistent with galaxies only, the Boötes I excess with stars only.

Not every Boötes I star is absorbed by the fit: ~25% of its excess sits at $(g-r)_0 < 0$ (blue
horizontal-branch stars and blue stragglers) or at the extreme of the error distribution, where neither
template has support. A complete model would add a template for these populations; for the question asked here —
stars or galaxies? — it does not matter.

Two cautions. The galaxy template describes field galaxies that leak into the SDSS star sample at 20 < r < 22.5;
a cluster at another redshift has different colours, and at some redshifts its red sequence crosses the stellar
locus — colours alone then fail. And this colour test does not use the CMD, which is the strongest evidence
that an excess is a stellar population at one distance (Chapter 2; Chapter 6).

## 3.3 The shape of the excess

The excess objects can also be compared with real stars in shape. For all SDSS objects (any type) with
20 < r < 22.5, take $\Delta m = r_{\rm PSF} - r_{\rm cModel}$, subtract the off-region distribution from the
on-region one, and compare with HST-confirmed stars and galaxies *of the same magnitude* from COSMOS (bright stars
would be a wrong reference: their Δm distribution is much narrower).

In [ ]:
a = load('a370_sdss')
R = np.hypot(*tangent(a.ra.to_numpy(), a.dec.to_numpy(), *A370)) * 60
r = (a.psfmag_r - a.extinction_r).to_numpy()
dm = (a.psfmag_r - a.cmodelmag_r).to_numpy()
s = (r > 20) & (r < 22.5)
db = np.arange(-0.4, 1.21, 0.1)
excess = np.histogram(dm[s & (R < 2)], db)[0] - 4 / (19.8**2 - 64) * np.histogram(dm[s & (R > 8) & (R < 19.8)], db)[0]
dmc = (cs.psfmag_r - cs.cmodelmag_r).to_numpy()
fc = matched & faint
fig, ax = plt.subplots(figsize=(7, 4.3))
frac = lambda h: h / h.sum()                                    # fraction of objects per 0.1-mag bin
ax.step(db[:-1], frac(excess), where='post', lw=2.5, c='k', label=f'Abell 370 excess (R < 2′), N = {excess.sum():.0f}')
ax.step(db[:-1], frac(np.histogram(dmc[fc & hst_star], db)[0]), where='post', lw=1.5, label='COSMOS HST stars')
ax.step(db[:-1], frac(np.histogram(dmc[fc & ~hst_star], db)[0]), where='post', lw=1.5, label='COSMOS HST galaxies')
ax.axvline(0.145, c='0.5', ls='--', lw=1)
ax.set(xlabel='$r_{\\rm PSF} - r_{\\rm cModel}$', ylabel='fraction per 0.1 mag', title='all SDSS objects, 20 < r < 22.5')
ax.legend(fontsize=8)
plt.tight_layout()
frac_ext = excess[db[:-1] >= 0.2].sum() / excess.sum()
print(f'fraction with Δm >= 0.2: Abell 370 excess {frac_ext:.2f}; COSMOS HST stars {np.mean(dmc[fc & hst_star] >= 0.2):.2f}; '
      f'COSMOS HST galaxies {np.mean(dmc[fc & ~hst_star] >= 0.2):.2f}')
assert frac_ext > 2 * np.mean(dmc[fc & hst_star] >= 0.2)

**Figure 3.4.** Nearly half of the Abell 370 excess objects have Δm ≥ 0.2, against ~10% of faint stars: as a
population they are extended objects, even though many of them individually pass as stars (Δm < 0.145). The
cluster members are also somewhat more compact than typical field galaxies of the same magnitude.

## Summary

- A compact, highly significant overdensity of "point sources" can be a galaxy cluster.
- Test the **excess** population, with the background subtracted: its colours (two-template likelihood for
  $f_\star$), its shapes compared with stars of the same magnitude, and — where available — a deeper survey.
- Build templates with the photometric errors of the target, including their correlation between colours.
- Colour templates are redshift dependent; the CMD (a turn-off, a giant branch, a horizontal branch at one
  distance) remains the strongest evidence for a stellar system.